<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.3-fine-tuning-with-vertex-ai/notebooks/exercises-7_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.3: Fine-Tuning on Vertex AI

*Module 7 · Fine-Tuning LLMs LIVE CLASS*

> Vertex AI lets you fine-tune Gemini models with your own data — no GPU management, no infrastructure, no training loops. Upload JSONL, configure hyperparameters, click train. Your custom model deploys as an endpoint you call like any other Gemini model. This lesson walks through the entire process in Python.

`Vertex AI` · `Gemini Tuning` · `Supervised` · `Evaluation` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-7.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Prepare & Upload Training Data — `01_prepare_upload.py`
2. Step 2 — Launch Fine-Tuning Job — The Core API Call — `02_launch_tuning.py`
3. Step 3 — Use Your Fine-Tuned Model — `03_use_tuned_model.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q datasets


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Prepare & Upload Training Data

Format data as Gemini expects it, upload to Google Cloud Storage.

**`01_prepare_upload.py`** · _Block 1 of 3_

PREPARE AND UPLOAD TRAINING DATA TO GCS — pip install google-cloud-storage google-cloud-aiplatform


In [ ]:
# PREPARE AND UPLOAD TRAINING DATA TO GCS
# pip install google-cloud-storage google-cloud-aiplatform
import json

# ── Step 1: Format data for Gemini fine-tuning ──
# Vertex AI Gemini expects this exact structure:
training_data = [
    {"contents": [
        {"role": "user", "parts": [{"text": "What is the refund policy?"}]},
        {"role": "model", "parts": [{"text": "Full refund within 7 days. 50% from 8-30 days. None after 30 days. Contact [email protected]."}]}
    ]},
    {"contents": [
        {"role": "user", "parts": [{"text": "How much does the GenAI course cost?"}]},
        {"role": "model", "parts": [{"text": "The GenAI Complete Course costs 14,999 rupees. It includes lifetime access, Discord community, weekly live sessions, certificate, and 5,000 rupees of GCP credits."}]}
    ]},
    {"contents": [
        {"role": "user", "parts": [{"text": "What are the prerequisites?"}]},
        {"role": "model", "parts": [{"text": "Basic Python programming and high school math. No ML experience needed. We teach everything from scratch."}]}
    ]},
]

# Save as JSONL
with open("train.jsonl", "w") as f:
    for item in training_data:
        f.write(json.dumps(item) + "\n")

print("Vertex AI Gemini Fine-Tuning Data Format:\n")
print("  Key differences from OpenAI format:")
print("    OpenAI: 'messages' with 'role': 'assistant'")
print("    Gemini: 'contents' with 'role': 'model', 'parts': [{'text':...}]")
print(f"\n  Saved {len(training_data)} examples to train.jsonl")

# ── Step 2: Upload to GCS ──
print(f"\n  Upload to GCS:")
print(f"    gsutil cp train.jsonl gs://YOUR_BUCKET/finetune/train.jsonl")
print(f"    gsutil cp val.jsonl gs://YOUR_BUCKET/finetune/val.jsonl")

# Or via Python:
# from google.cloud import storage
# client = storage.Client()
# bucket = client.bucket("YOUR_BUCKET")
# blob = bucket.blob("finetune/train.jsonl")
# blob.upload_from_filename("train.jsonl")


> **What just happened?** Training data formatted in Gemini’s specific structure: contents with role: "model" (not “assistant”) and parts: [{text: "..."}] (not just a string). Saved as JSONL and ready for GCS upload. Format matters: wrong format = job fails. Always validate against the Vertex AI docs.


### Step 2 · Launch Fine-Tuning Job — The Core API Call

One function call to start training. Vertex AI handles everything else.

**`02_launch_tuning.py`** · _Block 2 of 3_

LAUNCH GEMINI FINE-TUNING ON VERTEX AI


In [ ]:
# LAUNCH GEMINI FINE-TUNING ON VERTEX AI
import vertexai
from vertexai.tuning import sft as sft_train
import os, time

# Initialize Vertex AI
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "your-project")
REGION = "us-central1"
vertexai.init(project=PROJECT_ID, location=REGION)

# ── Launch supervised fine-tuning ──
tuning_job = sft_train.train(
    source_model="gemini-2.0-flash-001",
    train_dataset="gs://YOUR_BUCKET/finetune/train.jsonl",
    validation_dataset="gs://YOUR_BUCKET/finetune/val.jsonl",  # optional
    epochs=3,                    # 2-5 for most tasks
    learning_rate_multiplier=1.0, # 0.1-2.0, start with 1.0
    tuned_model_display_name="netsetos-support-v1",
)

print("Fine-Tuning Job Launched!\n")
print(f"  Model: gemini-2.0-flash-001")
print(f"  Epochs: 3")
print(f"  Job name: {tuning_job.name}")

# ── Monitor progress ──
print(f"\n  Monitoring:")
# while not tuning_job.has_ended:
#     time.sleep(60)
#     tuning_job.refresh()
#     print(f"    Status: {tuning_job.state}")

print(f"    Typical training time: 1-4 hours for 500-5000 examples")
print(f"    Cost: ~$0.50-$5 for small datasets on Flash")
print(f"\n  When complete:")
print(f"    tuning_job.tuned_model_name  # your custom model endpoint")
print(f"    tuning_job.tuned_model_endpoint_name  # call like any model")


### Step 3 · Use Your Fine-Tuned Model

Call your custom model exactly like the base model. Same API, your knowledge.

**`03_use_tuned_model.py`** · _Block 3 of 3_

USE YOUR FINE-TUNED MODEL


In [ ]:
# USE YOUR FINE-TUNED MODEL
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project="your-project", location="us-central1")

# ── Load your fine-tuned model ──
# Replace with your actual tuned model endpoint
TUNED_MODEL = "projects/YOUR_PROJECT/locations/us-central1/endpoints/YOUR_ENDPOINT"

# Option A: via endpoint name from tuning job
# model = GenerativeModel(model_name=tuning_job.tuned_model_endpoint_name)

# Option B: via model name
# model = GenerativeModel(model_name=tuning_job.tuned_model_name)

# ── Compare: base model vs fine-tuned ──
base_model = GenerativeModel("gemini-2.0-flash")
# tuned_model = GenerativeModel(model_name=TUNED_MODEL)

test_queries = [
    "What is the refund policy?",
    "How much does the GenAI course cost?",
    "Do you offer EMI options?",
    "What are the prerequisites?",
]

print("Base Model vs Fine-Tuned Model:\n")
for q in test_queries:
    base_resp = base_model.generate_content(q)
    # tuned_resp = tuned_model.generate_content(q)
    print(f"  Q: {q}")
    print(f"  Base:  {base_resp.text.strip()[:100]}")
    # print(f"  Tuned: {tuned_resp.text.strip()[:100]}")
    print(f"  Tuned: [will match Netsetos tone, include specific details]\n")

print("Expected improvements after fine-tuning:")
print("  1. Consistent tone matching your brand")
print("  2. Accurate company-specific details without RAG")
print("  3. Follows your response format/style")
print("  4. Handles domain jargon naturally")


> **What just happened?** Evaluation is the step most teams skip — and the one that determines whether fine-tuning was worth it. Always measure on held-out data the model never saw during training . If the tuned model isn't significantly better on your specific task, prompt engineering or RAG may be sufficient (revisit Lesson 7.1).


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
